# 3b. External data

Reads files that never went through a questionnaire - published indicator
tables from another source, one folder per chapter under
`DATA COLLECTOR\external data\<Chapter>\` - and appends them to that
chapter's long files.

```
DATA COLLECTOR\external data\<Chapter>\*.xlsx
        -> merged_long_files\<Chapter>_EN.xlsx   (direct - these files are English)
        -> merged_long_files\<Chapter>_AR.xlsx   (once every value has an Arabic form)
```

**Runs after notebook 3, on the English side** - these files are already in
English, so there is nothing to translate on the way in. What needs Arabic is
the small, closed set of *labels* a file introduces - a country, an
indicator name, a citation - not the data itself, and that goes through the
dictionary as its own step below.

## Why interactive, not a script

Two things here cannot be done safely without a person looking at them, and
both are exactly why this stayed a notebook rather than becoming a script
the first time it worked:

- **Structure inference.** Nothing here assumes "row 1 is headers." The one
  real file used to build this had six sheets and three different header
  shapes - single row, a group row above it, a group row built into the
  anchor row itself with the split one row below - discovered by reading the
  actual file, not by guessing at a spec. A sheet this cannot confidently
  read is refused and logged, never guessed at - `Compendium_5_Charts.ipynb`
  and `4.10` in the sample file both know why: one stray value with no
  header above it anywhere, entered one column too far right in the source
  file. Silently dropping it would have been wrong; silently keeping it
  under an invented header would have been worse.
- **Every new label needs a person's word once.** A country, indicator, or
  citation this has never seen gets its Arabic supplied by a person (Claude,
  reviewing the run), recorded in the dictionary, and reused automatically
  from then on - the same "Filling dictionary gaps" loop notebook 3 already
  uses for calculated labels, in `../CLAUDE.md`. A close-enough existing
  spelling is reused instead of asked about again - `Comoros` against the
  dictionary's own `Comoros Islands` - but always reported, never silent:
  a good score is not certainty, and a wrong reuse would quietly merge two
  different things into one row everywhere downstream. Indicator and Source
  are never fuzzy-matched at all, only ever exact, for the same reason
  Source already wasn't in notebook 1: a long, templated sentence carries
  its topic in a small fraction of the string, so two different indicators
  sharing a structure can still score above the cutoff. Not a guess - this
  project's own health-spending text scored 0.628 against its
  education-spending text on nothing but shared wording.

## Marking what came from outside

Every row this appends gets `Data Origin` = `External` (`مصدر البيانات` =
`بيانات خارجية` on the Arabic side) - blank for every row that was already
there. Notebook 4's row and breakdown columns are a fixed, named list
(`ROW_COLUMNS`, `BREAKDOWN_COLUMNS`); `Data Origin` is in neither, so
tabulations already ignore it without anything there having to change.

## Running it

1. Run the cells through **Run - part 1** below. It appends every chapter's
   external files to the English long file and reports what it found:
   sheets it read, sheets it refused (and why), and every label that needs
   an Arabic form it does not already have - `GAPS`.
2. Fill in `GAPS["val_ar"]`, by position - never by retyping the English or
   Arabic, the same rule the rest of this project's gap-filling follows.
   `update_dictionary(filled)` backs the dictionary up first.
3. Call **`apply_gaps()`**. It rebuilds the appended rows with their Arabic
   form and writes them to the Arabic long file.

Idempotent throughout: `Data Origin == "External"` is exactly what marks a
row this added, so re-running strips its own previous rows before adding the
fresh ones - nothing duplicates on a second run.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
from collections import defaultdict
import logging
import re
from pathlib import Path

import openpyxl
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config

In [ ]:
"""
CELL: Configuration - paths, the chapters this looks for, and the new column
that marks a row as coming from outside the questionnaires.
"""
DATA_COLLECTOR_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "DATA COLLECTOR"
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
EXTERNAL_DATA_PATH = DATA_COLLECTOR_PATH / "external data"
COMPENDIUM_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "COMPENDIUM-ARAB SOCIETY"
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"

# Leave as None to check every chapter with an external-data subfolder, or
# restrict e.g. ["Health"].
CHAPTERS = None

# Same number and same meaning as notebook 1's: a value just above this line
# is trustworthy enough for the real pipeline to fix on its own without
# review, so it is trustworthy enough here too - see resolve_arabic().
FUZZY_MATCH_CUTOFF = 0.6

# Never fuzzy-matched, only ever exact - same idea as notebook 1's Source
# column, and Indicator turned out to need it for the same reason: a long,
# templated sentence carries its topic in a small fraction of the string,
# so two different indicators sharing a structure can still score above the
# cutoff. Measured, not guessed - see resolve_arabic().
NEVER_FUZZY_MATCHED = {"Source", "Indicator"}

# Year and Value never need a value translated - a year is a year, a number
# is a number - but the COLUMN NAME still has to match notebook 1's own
# constants of the same name, or the Arabic file ends up with literal
# English column headers that notebook 4's tabulations cannot find: it looks
# for "السنة" and "العدد" by name, not by position.
YEAR_COLUMN_AR = "السنة"
VALUE_COLUMN_AR = "العدد"

# The six domains - both here and as `external data\<name>\` subfolder names.
KNOWN_CHAPTERS = ["Health", "Population", "Education", "Labor", "Poverty", "Housing"]

# Marks a row as not from the questionnaires. Blank for every existing row -
# nothing already in a long file is touched or backfilled with this - and set
# only on rows this notebook itself appends. Not in notebook 4's ROW_COLUMNS
# or BREAKDOWN_COLUMNS, so tabulations never see it; nothing there has to
# change for that to stay true.
ORIGIN_COLUMN_EN = "Data Origin"
ORIGIN_COLUMN_AR = "مصدر البيانات"
ORIGIN_EXTERNAL_EN = "External"
ORIGIN_EXTERNAL_AR = "بيانات خارجية"

# One file per chapter - pipeline_inconsistencies_<Chapter>.txt - plus
# pipeline_inconsistencies_general.txt for anything with no chapter of its
# own. See save_inconsistencies() below.


def chapter_log_path(chapter):
    """Where one chapter's own findings live - every notebook's sections
    side by side, but never mixed with another chapter's."""
    return COMPENDIUM_PATH / f"pipeline_inconsistencies_{chapter}.txt"


GENERAL_LOG_PATH = COMPENDIUM_PATH / "pipeline_inconsistencies_general.txt"


def _write_section(path, title, section, section_text):
    """Replace one named section in one file, in place, leaving every other
    section exactly as it was. The mechanic every chapter file and the
    general one share - split what's there into sections by name, replace
    this one, rebuild in section order so the file reads the same whatever
    order the notebooks last ran in."""
    header = [title, "=" * 78, ""]

    sections = {}
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body_text = "\n\n".join(sections[name] for name in sorted(sections))
    path.write_text("\n".join(header).rstrip("\n") + "\n\n" + body_text + "\n",
                    encoding="utf-8")


def save_inconsistencies(section, records, chapters=None):
    """Write this notebook's findings for this section, one file per chapter
    - `pipeline_inconsistencies_<Chapter>.txt`, beside the codes folder - so
    a chapter-scoped run of one notebook never overwrites a different
    chapter's history the way one shared file used to. A finding with no
    chapter (a dictionary-level problem, not a source-data one) goes to
    `pipeline_inconsistencies_general.txt` instead.

    `chapters` is every chapter this call actually covers, independent of
    whether any of them have a finding - pass it explicitly so a clean
    chapter still gets its section correctly replaced with "Nothing found"
    rather than left showing whatever an earlier, unrelated run left there.
    Inferred from `records` if not given, which undercounts a chapter with
    zero findings; every call site in this pipeline passes it explicitly.
    """
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    marker = f"### {section} ###"

    WHERE = ["country", "indicator", "year", "sex", "age_group",
             "nationality", "area", "file", "sheet", "row"]

    def render(chapter_records):
        body = [marker, f"    last run {stamp}", ""]
        if not chapter_records:
            body += ["    Nothing found.", ""]
        else:
            frame = pd.DataFrame(chapter_records)
            for kind, group in frame.groupby("kind", sort=False):
                body.append(f"  {kind.upper()}  ({len(group)})")
                for _, row in group.iterrows():
                    def show(value):
                        if isinstance(value, float) and float(value).is_integer():
                            return str(int(value))
                        return str(value)
                    where = " · ".join(
                        show(row[f]) for f in WHERE
                        if f in row and pd.notna(row[f]) and str(row[f]) != "")
                    body.append(f"      {where}" if where else "      -")
                    body.append(f"          {row['detail']}")
                body.append("")
        return "\n".join(body)

    by_chapter = defaultdict(list)
    general = []
    for record in records:
        chapter = record.get("chapter")
        if chapter in (None, "", "-"):
            general.append(record)
        else:
            by_chapter[str(chapter)].append(record)

    covered = {str(c) for c in (chapters or [])} | set(by_chapter)

    paths = []
    for chapter in sorted(covered):
        path = chapter_log_path(chapter)
        _write_section(path, f"PIPELINE INCONSISTENCIES — {chapter}", section,
                       render(by_chapter.get(chapter, [])))
        paths.append(path)

    if general or not covered:
        _write_section(GENERAL_LOG_PATH, "PIPELINE INCONSISTENCIES — general", section,
                       render(general))
        paths.append(GENERAL_LOG_PATH)

    return paths, len(records)


def external_chapters():
    """Chapters with a folder under external data\\ holding at least one .xlsx."""
    if not EXTERNAL_DATA_PATH.exists():
        logger.warning(f"{EXTERNAL_DATA_PATH} does not exist - nothing to read")
        return []
    found = []
    names = CHAPTERS if CHAPTERS else KNOWN_CHAPTERS
    for name in names:
        folder = EXTERNAL_DATA_PATH / name
        if folder.exists() and any(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$")):
            found.append(name)
    return found


## The dictionary, both directions

In [ ]:
"""
CELL: The dictionary, read both directions.

Everywhere else in this project the dictionary is read Arabic to English only
- CLAUDE.md is explicit that nothing should build the reverse map, because
inverting it is lossy exactly where several Arabic spellings share one
English translation (43 such terms on the current data), and notebook 3 was
once rewritten specifically to stop doing that.

This is different in kind, not a quiet exception to that rule: it is not
translating existing, already-correct rows backward - it is giving a brand
new English value, one that has never had an Arabic form before, an Arabic
form for the first time. The English side is looked up exactly, and reused
if the exact same (column, value) pair has ever been recorded - only a value
with no recorded Arabic at all becomes a gap for a person to fill, the same
place calculated-label gaps already go in notebook 3.
"""


def load_dictionary():
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}          # Arabic -> English, as elsewhere
    english_column_map = {}                  # English -> Arabic column name
    english_value_map = {}                   # English column -> {English value: Arabic value}

    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        english_column = rows["col_en"].iloc[0]
        column_map[arabic_column] = english_column
        value_map[arabic_column] = {
            ar: en for ar, en in zip(rows["val_ar"], rows["val_en"]) if pd.notna(ar)
        }

    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        arabic_column = rows["col_ar"].iloc[0] if rows["col_ar"].notna().any() else None
        english_column_map[english_column] = arabic_column
        english_value_map[english_column] = {
            str(en).strip(): ar
            for en, ar in zip(rows["val_en"], rows["val_ar"])
            if pd.notna(en) and pd.notna(ar)
        }

    return (column_map, value_map), (english_column_map, english_value_map)


DICTIONARY_AR_TO_EN, DICTIONARY_EN_TO_AR = load_dictionary()
logger.info(f"Dictionary loaded: {len(DICTIONARY_AR_TO_EN[0])} column(s), both directions")


def arabic_for_column(english_column):
    """The Arabic name for an English column, adding it to the running map
    (not the file - that only happens through update_dictionary()) the first
    time this notebook itself invents one, so a second lookup in the same run
    reuses it instead of asking again."""
    english_column_map, _ = DICTIONARY_EN_TO_AR
    return english_column_map.get(english_column)


def arabic_for_value(english_column, english_value):
    """The Arabic value already on file for this exact (column, value) pair,
    or None if this notebook has never seen it before."""
    _, english_value_map = DICTIONARY_EN_TO_AR
    return english_value_map.get(english_column, {}).get(str(english_value).strip())


def best_match(text, choices):
    """The known choice difflib considers closest to text, and how close (0
    to 1) - identical to notebook 1's function of the same name."""
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def resolve_arabic(english_column, english_value):
    """The one place every gap-or-not decision is made, so find_gaps() and
    translate_to_arabic() can never disagree with each other.

    Returns (val_ar, method): method is "exact" for a value already on file
    under this exact spelling, "fuzzy" for a close match reused from a
    DIFFERENT spelling already on file (a country's long form against its
    short form - "State of Palestine" against "Palestine", score 0.667 -
    reusing its Arabic rather than filing a second, inconsistent entry that
    would split one country across two rows in every AR tabulation), or
    (None, None) if nothing on file is close enough and this is a real gap.

    The same FUZZY_MATCH_CUTOFF the rest of the pipeline uses, for short
    categorical values like Country - a value just above the line there is
    trustworthy enough for the real pipeline to fix on its own without
    review, so it is trustworthy enough here too.

    NEVER_FUZZY_MATCHED columns skip straight from exact to "no match, real
    gap" - proven necessary the same way Source already was, empirically:
    "Government expenditure on health as % of Gross Domestic Product (GDP)"
    scored 0.628 against the dictionary's *education* spending indicator,
    over the cutoff, on nothing but shared sentence template. A long,
    templated Indicator sentence carries its distinguishing word in a small
    fraction of the string, so a wrong topic can still look close enough -
    unlike Country or Sex, where the whole string more or less is the topic.
    """
    exact = arabic_for_value(english_column, english_value)
    if exact is not None:
        return exact, "exact"
    if english_column in NEVER_FUZZY_MATCHED:
        return None, None

    _, english_value_map = DICTIONARY_EN_TO_AR
    known = english_value_map.get(english_column, {})
    if not known:
        return None, None
    match, score = best_match(str(english_value).strip(), known.keys())
    if match is not None and score >= FUZZY_MATCH_CUTOFF:
        return known[match], "fuzzy"
    return None, None


## Structure inference - title, header shape, forward-fill

In [ ]:
"""
CELL: infer_sheet() - work out one raw sheet's shape and turn it into long rows.

Nothing here is a fixed "row 1 is headers" assumption - every real sheet in
the first external file broke that assumption in a different way. What is
reliable, across every shape seen so far, is that the row holding "Country"
and "Year" in its first two cells anchors everything else - whether or not
there is a group-label row, and whichever side of the anchor it sits on:

  - group row ABOVE the anchor, anchor row itself holds the leaf labels
    (e.g. a "Population per one personnel" / "Health Personnel per 100,000"
    row above "Country, Year, Number of physicians, Number of dentists, ...");
  - group row ON the anchor row, leaf labels BELOW it
    (e.g. "Country, Year, Visual, Hearing, ..." then a "Male, Female, Total"
    row underneath each group);
  - no group at all - the anchor row's own cells are the leaf labels.

A merged group cell (openpyxl only keeps a value in the top-left cell of a
merge) is forward-filled rightward before use.

Confidence is refused, not guessed, when no row has "Country" then "Year" in
its first two cells at all, or when a column has real data below it but no
label a plan could be built for. A skip is logged with the reason, never
silently dropped.
"""

SEX_LABELS = {"male": "Male", "female": "Female", "total": "Both sexes"}


def is_blank(value):
    return value is None or (isinstance(value, str) and value.strip() == "")


def clean_text(value):
    return "" if is_blank(value) else str(value).strip()


def find_header_anchor(ws, max_scan_row=15):
    """The row where column A reads 'Country' and column B reads 'Year',
    case-insensitively - or None if no such row exists in the first
    max_scan_row rows.
    """
    for row in range(1, max_scan_row + 1):
        a = clean_text(ws.cell(row=row, column=1).value).lower()
        b = clean_text(ws.cell(row=row, column=2).value).lower()
        if a == "country" and b == "year":
            return row
    return None


def read_row(ws, row, last_col):
    """Row values as a 0-indexed list, position i = column i+1. Position 0 is
    always Country, position 1 is always Year, for a row read at the anchor."""
    if row < 1:
        return [None] * last_col
    return [ws.cell(row=row, column=c).value for c in range(1, last_col + 1)]


def forward_fill_right(values):
    """Blank cells inherit the last non-blank value to their left - how a
    merged header cell reads back once openpyxl un-merges it (only the
    top-left cell of a merge keeps its value; the rest come back as None).
    """
    filled, last = [], None
    for value in values:
        if not is_blank(value):
            last = value
        filled.append(last)
    return filled


def build_column_plan(ws, anchor_row, last_col):
    """What each column beyond Country/Year actually is, whichever of the
    three shapes described above this sheet turns out to be.

    Returns (plan, data_start_row), or (None, reason) if no shape here could
    be told apart confidently - `plan` is a list aligned to columns 3..last_col,
    each entry {"indicator": str or None, "sex": str or None, "use_title": bool}
    or None where a column carries nothing.
    """
    anchor_values = read_row(ws, anchor_row, last_col)
    above_values = read_row(ws, anchor_row - 1, last_col)
    below_values = read_row(ws, anchor_row + 1, last_col)

    above_has_content = any(not is_blank(v) for v in above_values[2:])
    below_has_content = any(not is_blank(v) for v in below_values[2:])

    if above_has_content:
        group_values = forward_fill_right(above_values)
        leaf_values = anchor_values
        data_start_row = anchor_row + 1
    elif below_has_content:
        group_values = forward_fill_right(anchor_values)
        leaf_values = below_values
        data_start_row = anchor_row + 2
    else:
        group_values = [None] * last_col
        leaf_values = anchor_values
        data_start_row = anchor_row + 1

    # A blank buffer row (or more) between the header block and the data is
    # common (4.7 has two). Skip forward past any fully-empty rows.
    max_row = ws.max_row
    while data_start_row <= max_row and all(
        is_blank(v) for v in read_row(ws, data_start_row, last_col)
    ):
        data_start_row += 1

    plan = []
    for position in range(2, last_col):  # columns 3..last_col
        sub_label = clean_text(leaf_values[position])
        group_label = clean_text(group_values[position])
        if sub_label == "" and group_label == "":
            plan.append(None)
            continue

        sub_key = sub_label.lower()
        if sub_key in SEX_LABELS:
            # The title always supplies the base name here, whether or not
            # there is a group - a bare "Visual" would become its own
            # tabulation sheet in notebook 4, one per Indicator, with no
            # "disability" anywhere in the name it appears under.
            plan.append({
                "indicator": group_label or None,
                "sex": SEX_LABELS[sub_key],
                "use_title": True,
            })
        else:
            name = f"{group_label} - {sub_label}" if group_label and sub_label else (sub_label or group_label)
            plan.append({"indicator": name, "sex": None, "use_title": False})

    return plan, data_start_row


## extract_sheet() - one sheet's plan, turned into rows

In [ ]:
"""
CELL: extract_sheet() - turn one raw sheet into long rows, or refuse it.
"""


def read_title(ws, anchor_row):
    """The title above the header block, if there is one - row 1's own text,
    with a leading "Table 4.7" - style prefix dropped since the table number
    means nothing outside this workbook."""
    if anchor_row <= 1:
        return None
    text = clean_text(ws.cell(row=1, column=1).value)
    if not text:
        return None
    return re.sub(r"^Table\s+\S+\s*", "", text, flags=re.I).strip() or text


def extract_sheet(ws, sheet_name, file_name):
    """Returns (rows, note) - `rows` is a list of dicts ready to become a
    DataFrame, empty if this sheet was refused; `note` explains what happened
    either way, for the run's report.
    """
    anchor_row = find_header_anchor(ws)
    if anchor_row is None:
        return [], f"no 'Country'/'Year' header found in the first 15 rows - skipped"

    last_col = ws.max_column
    plan, data_start_row = build_column_plan(ws, anchor_row, last_col)
    title = read_title(ws, anchor_row)

    if all(p is None for p in plan):
        return [], "header row found but no column could be named - skipped"
    if any(p and p["use_title"] and not title for p in plan):
        return [], "a Male/Female/Total column with no group and no title to name it by - skipped"

    # Confidence check: every column with real numeric data below it must
    # have a plan entry. A column with data but no name means the shape was
    # not understood correctly, not that the column is genuinely empty -
    # refuse the whole sheet rather than silently dropping a column of data.
    unnamed_but_populated = []
    for position in range(2, last_col):
        if plan[position - 2] is not None:
            continue
        column = position + 1
        for row in range(data_start_row, ws.max_row + 1):
            if not is_blank(ws.cell(row=row, column=column).value):
                unnamed_but_populated.append(column)
                break
    if unnamed_but_populated:
        return [], (f"column(s) {unnamed_but_populated} have data but no header this could "
                    f"name - skipped rather than dropping them silently")

    rows = []
    current_country = None
    for row in range(data_start_row, ws.max_row + 1):
        raw_country = ws.cell(row=row, column=1).value
        if not is_blank(raw_country):
            current_country = clean_text(raw_country)
        raw_year = ws.cell(row=row, column=2).value
        year = pd.to_numeric(raw_year, errors="coerce")
        if pd.isna(year) or current_country is None:
            continue  # a blank trailer row, a footnote line, or data before any country appeared

        for position in range(2, last_col):
            entry = plan[position - 2]
            if entry is None:
                continue
            column = position + 1
            value = ws.cell(row=row, column=column).value
            number = pd.to_numeric(value, errors="coerce")
            if pd.isna(number):
                continue  # blank cell for this indicator/year - not a finding, just unreported

            if entry["use_title"]:
                indicator = f"{title} - {entry['indicator']}" if entry["indicator"] else title
            else:
                indicator = entry["indicator"]
            rows.append({
                "Country": current_country, "Year": int(year), "Value": float(number),
                "Indicator": indicator, "Sex": entry["sex"],
                "Source": f"{file_name}, sheet {sheet_name}",
            })

    if not rows:
        return [], "header understood but no usable data rows found - skipped"
    return rows, f"{len(rows):,} row(s) extracted, {sum(1 for p in plan if p)} indicator column(s)"


## Reading a chapter's external-data folder

In [ ]:
"""
CELL: read_external_file() - every sheet in one workbook, extracted or refused.
"""

# Categorical columns that need an Arabic form - Year and Value are numbers,
# used as-is in both languages, and never have a dictionary entry.
TRANSLATABLE_COLUMNS = ["Country", "Indicator", "Sex", "Source"]

SKIPPED_SHEETS = []   # {chapter, file, sheet, detail} - every sheet this refused


def read_external_file(path, chapter):
    """Every usable row from every sheet in one external-data workbook."""
    try:
        wb = openpyxl.load_workbook(path, data_only=True)
    except Exception as error:
        SKIPPED_SHEETS.append({
            "chapter": chapter, "file": path.name, "sheet": "-",
            "detail": f"could not open the file: {type(error).__name__}: {error}",
        })
        return []

    all_rows = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows, note = extract_sheet(ws, sheet_name, path.name)
        logger.info(f"  {path.name} | {sheet_name}: {note}")
        if not rows:
            SKIPPED_SHEETS.append({
                "chapter": chapter, "file": path.name, "sheet": sheet_name, "detail": note,
            })
            continue
        all_rows.extend(rows)
    return all_rows


def read_external_chapter(chapter):
    """Every usable row from every .xlsx in one chapter's external-data folder."""
    folder = EXTERNAL_DATA_PATH / chapter
    files = sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$"))
    logger.info(f"{chapter}: {len(files)} external file(s)")

    rows = []
    for path in files:
        rows.extend(read_external_file(path, chapter))
    if not rows:
        return pd.DataFrame(columns=["Country", "Year", "Value", "Indicator", "Sex", "Source"])

    table = pd.DataFrame(rows)
    table["Chapter"] = chapter
    table[ORIGIN_COLUMN_EN] = ORIGIN_EXTERNAL_EN
    return table


## Appending, and the gap-and-apply loop

In [ ]:
"""
CELL: Appending - to the English file directly, to the Arabic file once the
dictionary has an entry for everything new.
"""


def strip_previous_external(table, chapter):
    """Remove this chapter's own previously-appended external rows before
    adding fresh ones, the same idempotent pattern notebook 2's
    build_chapter() uses for its calculated rows - running this notebook
    twice must not duplicate anything."""
    if ORIGIN_COLUMN_EN not in table.columns and ORIGIN_COLUMN_AR not in table.columns:
        return table
    origin_column = ORIGIN_COLUMN_EN if ORIGIN_COLUMN_EN in table.columns else ORIGIN_COLUMN_AR
    marker = ORIGIN_EXTERNAL_EN if origin_column == ORIGIN_COLUMN_EN else ORIGIN_EXTERNAL_AR
    chapter_column = "Chapter" if "Chapter" in table.columns else None
    is_previous = table[origin_column] == marker
    if chapter_column:
        is_previous &= table[chapter_column].astype(str).str.strip() == chapter
    removed = int(is_previous.sum())
    if removed:
        logger.info(f"  {chapter}: removing {removed:,} external row(s) from a previous run")
    return table[~is_previous]


def append_to_english(chapter, new_rows):
    """Adds new_rows to <Chapter>_EN.xlsx in place. Creates the file if this
    chapter has never been run - notebook 3 will still build the real one
    from the questionnaires the next time it runs, this just means the
    external rows do not have to wait for that."""
    path = LONG_FILES_PATH / f"{chapter}_EN.xlsx"
    existing = pd.read_excel(path, engine="openpyxl") if path.exists() else pd.DataFrame()
    existing = strip_previous_external(existing, chapter)

    combined = pd.concat([existing, new_rows], ignore_index=True)
    combined.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"  {chapter}: {len(new_rows):,} external row(s) -> {path.name} "
               f"({len(combined):,} rows)")
    return combined


FUZZY_MATCHES = []   # every value resolved by a close-enough match, not an exact one


def find_gaps(new_rows):
    """Every (column, value) this notebook needs an Arabic form for and does
    not already have one - the chapter name and the Data Origin marker
    included, since those need translating exactly once too.

    A close-enough fuzzy match against an existing English spelling is
    reused rather than filed as a gap ("State of Palestine" against the
    dictionary's own "Palestine"), but is always reported in FUZZY_MATCHES -
    a good score is not certainty, and a wrong reuse would quietly merge two
    different things into one row everywhere downstream.
    """
    gaps, fuzzy = [], []
    seen = set()

    def check(col_en, val_en):
        val_en = str(val_en).strip()
        key = (col_en, val_en)
        if key in seen or val_en == "" or val_en.lower() == "nan":
            return
        seen.add(key)
        resolved, method = resolve_arabic(col_en, val_en)
        if resolved is None:
            gaps.append({
                "col_en": col_en, "val_en": val_en,
                "col_ar": arabic_for_column(col_en),   # None if the column itself is new too
            })
        elif method == "fuzzy":
            fuzzy.append({"col_en": col_en, "val_en": val_en, "matched_val_ar": resolved})

    for column in TRANSLATABLE_COLUMNS:
        if column not in new_rows.columns:
            continue
        for value in new_rows[column].dropna().unique():
            check(column, value)

    for chapter in new_rows.get("Chapter", pd.Series(dtype=str)).dropna().unique():
        check("Chapter", chapter)

    FUZZY_MATCHES.extend(fuzzy)
    return pd.DataFrame(gaps, columns=["col_en", "val_en", "col_ar"])


def dictionary_key(col_ar, val_ar):
    """Normalizes NaN to None first - float('nan') never equals another
    float('nan'), so leaving it as-is would make a column-only row (see
    below) look "new" every time and duplicate itself on a second call."""
    return (col_ar, val_ar if pd.notna(val_ar) else None)


def update_dictionary(filled, backup=True):
    """Appends reviewed translations to translation dict.xlsx - identical to
    notebook 3's function of the same name, column-only-row fix included.
    Kept as its own copy since nothing is shared between notebooks here.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    candidates = filled[needed].copy()
    is_column_row = candidates["val_ar"].isna() & candidates["val_en"].isna()
    column_rows = candidates[is_column_row
                             & candidates["col_ar"].notna() & candidates["col_en"].notna()]

    value_rows = candidates[~is_column_row].dropna()
    value_rows = value_rows[(value_rows["val_ar"].astype(str).str.strip() != "")
                            & (value_rows["val_en"].astype(str).str.strip() != "")]

    new_rows = pd.concat([column_rows, value_rows], ignore_index=True)
    if new_rows.empty:
        logger.warning("No completed rows to add.")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already_there = {dictionary_key(c, v)
                     for c, v in zip(dictionary["col_ar"], dictionary["val_ar"])}
    to_add = new_rows[~new_rows.apply(
        lambda r: dictionary_key(r["col_ar"], r["val_ar"]) in already_there, axis=1)]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    to_add = to_add.reindex(columns=dictionary.columns)
    if "status" in to_add.columns:
        to_add["status"] = "updated"

    updated = pd.concat([dictionary, to_add], ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
               f"({len(dictionary):,} -> {len(updated):,}).")
    return updated


def translate_to_arabic(table):
    """The English rows this notebook appended, rebuilt with Arabic column
    names and values - only possible once every value find_gaps() reported
    has a dictionary entry, which update_dictionary() is what supplies."""
    global DICTIONARY_AR_TO_EN, DICTIONARY_EN_TO_AR
    DICTIONARY_AR_TO_EN, DICTIONARY_EN_TO_AR = load_dictionary()   # pick up what update_dictionary() just added

    result = table.copy()
    still_missing = []
    for column in TRANSLATABLE_COLUMNS + ["Chapter"]:
        if column not in result.columns:
            continue
        arabic_column = arabic_for_column(column)
        if arabic_column is None:
            still_missing.append(column)
            continue

        # Resolve each DISTINCT value once, not once per row - resolve_arabic()
        # can fall back to a fuzzy scan of every known value in the column, and
        # doing that per row rather than per distinct value turned a few
        # hundred lookups into several million on a 6,788-row sheet.
        lookup = {v: resolve_arabic(column, v)[0]
                 for v in result[column].dropna().unique()}
        mapped = result[column].map(lookup)
        unresolved = result[column].notna() & mapped.isna()
        if unresolved.any():
            still_missing.extend(
                f"{column}: {v!r}" for v in result.loc[unresolved, column].unique())
        result[column] = mapped
        result = result.rename(columns={column: arabic_column})

    if ORIGIN_COLUMN_EN in result.columns:
        result[ORIGIN_COLUMN_EN] = result[ORIGIN_COLUMN_EN].map(
            {ORIGIN_EXTERNAL_EN: ORIGIN_EXTERNAL_AR}).fillna(result[ORIGIN_COLUMN_EN])
        result = result.rename(columns={ORIGIN_COLUMN_EN: ORIGIN_COLUMN_AR})

    # No value to translate - a year is a year, a number is a number - but
    # notebook 4's tabulations look for "السنة" and "العدد" by name, not
    # position, so the columns still have to be renamed.
    result = result.rename(columns={"Year": YEAR_COLUMN_AR, "Value": VALUE_COLUMN_AR})

    if still_missing:
        raise ValueError(
            "still missing an Arabic form for: " + ", ".join(map(str, still_missing[:10]))
            + " - call find_gaps() again and update_dictionary() before retrying")
    return result


def append_to_arabic(chapter, ar_rows):
    """The Arabic twin of append_to_english() - same idempotent strip, same
    create-if-missing."""
    path = LONG_FILES_PATH / f"{chapter}_AR.xlsx"
    existing = pd.read_excel(path, engine="openpyxl") if path.exists() else pd.DataFrame()
    existing = strip_previous_external(existing, chapter)

    combined = pd.concat([existing, ar_rows], ignore_index=True)
    combined.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"  {chapter}: {len(ar_rows):,} external row(s) -> {path.name} "
               f"({len(combined):,} rows)")
    return combined


## Run - part 1: read, append to English, find the gaps

In [ ]:
"""
CELL: Main run, part 1 - read every chapter's external files, append to
English, and collect the translation gaps. Stop and review before calling
apply_gaps() below - nothing is written to Arabic, or to the dictionary,
until that is called.
"""
SKIPPED_SHEETS.clear()
FUZZY_MATCHES.clear()

chapters = external_chapters()
print(f"Chapters with external data: {chapters}\n")

APPENDED_EN = {}   # chapter -> the rows this run appended (needed by apply_gaps())
ALL_GAPS = []

for chapter in chapters:
    print(f"=== {chapter} ===")
    new_rows = read_external_chapter(chapter)
    if new_rows.empty:
        print(f"  nothing usable found\n")
        continue

    append_to_english(chapter, new_rows)
    APPENDED_EN[chapter] = new_rows

    gaps = find_gaps(new_rows)
    if not gaps.empty:
        gaps.insert(0, "chapter", chapter)
        ALL_GAPS.append(gaps)
    print()

GAPS = pd.concat(ALL_GAPS, ignore_index=True) if ALL_GAPS else pd.DataFrame(
    columns=["chapter", "col_en", "val_en", "col_ar"])

print("=" * 70)
print(f"{sum(len(v) for v in APPENDED_EN.values()):,} row(s) appended across "
     f"{len(APPENDED_EN)} chapter(s)")
print(f"{len(SKIPPED_SHEETS)} sheet(s) skipped - see below")
print(f"{len(GAPS)} value(s) need an Arabic translation before appending to "
     f"the Arabic files - see GAPS")
print(f"{len(FUZZY_MATCHES)} value(s) reused a close-enough existing Arabic "
     f"spelling instead - worth a glance, see FUZZY_MATCHES")

if FUZZY_MATCHES:
    print("\nFuzzy matches - reused, not filed as gaps:")
    for row in FUZZY_MATCHES:
        print(f"  [{row['col_en']}] {row['val_en']!r} -> {row['matched_val_ar']!r}")

if SKIPPED_SHEETS:
    print("\nSkipped:")
    for row in SKIPPED_SHEETS:
        print(f"  {row['chapter']} · {row['file']} · {row['sheet']}: {row['detail']}")

records = [{"kind": "sheet skipped", **row} for row in SKIPPED_SHEETS]
records += [{"kind": "needs Arabic translation", "chapter": row["chapter"],
            "detail": f"[{row['col_en']}] {row['val_en']!r} has no Arabic form on file yet"}
           for _, row in GAPS.iterrows()]
records += [{"kind": "reused a close Arabic spelling",
            "detail": f"[{row['col_en']}] {row['val_en']!r} -> {row['matched_val_ar']!r}"}
           for row in FUZZY_MATCHES]
paths, count = save_inconsistencies("3b. EXTERNAL DATA", records, chapters=chapters)
print(f"\n{count} inconsistency(ies) recorded across {len(paths)} file(s)")

if not GAPS.empty:
    print("\nGAPS:")
    print(GAPS.to_string(index=False))


## Run - part 2: call once GAPS is filled and update_dictionary() has run

In [ ]:
"""
CELL: Main run, part 2 - call once GAPS has been filled in and update_dictionary()
has been called. Builds the Arabic twin of every row appended above and
writes it to <Chapter>_AR.xlsx.

    filled = GAPS.copy()
    filled["val_ar"] = [...]            # by position, never by retyping val_en
    update_dictionary(filled)
    apply_gaps()
"""


def apply_gaps():
    if not APPENDED_EN:
        logger.warning("Nothing was appended this run - nothing to translate to Arabic.")
        return

    for chapter, new_rows in APPENDED_EN.items():
        ar_rows = translate_to_arabic(new_rows)
        append_to_arabic(chapter, ar_rows)

    print(f"\nArabic rows written for: {list(APPENDED_EN)}")
